# Hardpoint Breakaway Tests - Historical Trend 
**Author**: Noah Gonzalez  
**Date**: 2026-03-17

This notebook evaluates the hardpoint breakaway tests conducted from 2023 to the present.

The pipeline is organized into modules. Each cell contains function calls imported from specific modules. Additionally, these functions include various arguments, allowing the user to select different parameters based on their specific requirements.

All the `.py` scripts used in this analysis are located in the python folder.

> **Note**: To recreate the interactive dashboards, you must have the *[bokeh](https://bokeh.org)* library installed.


In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

Installation of the `tn082` package (found in this repository) is required to run the modules

In [ ]:
import importlib
import sys, os
import lsst.sitcom.tn082

from lsst.sitcom.tn082.scan_status import scan_days_with_tests_real_status
from lsst.sitcom.tn082.features import extract_features_for_groups
from lsst.sitcom.tn082.plots_bokeh import (
    bokeh_daily_stiffness_dashboard,
    save_dashboard_html,
)
from bokeh.io import output_notebook, show

In [ ]:
# To correctly display the dashboards, ensure that the bokeh and jupyter_bokeh extensions
# are installed and enabled.

output_notebook()

### Data Scanning and Processing (scan_status.py)

The `scan_status.py` module identifies the specific days when tests were performed, as tests are not conducted daily. Furthermore, the information is organized into groups to streamline and simplify the query process.

#### Available Arguments:

* **start_date**: The starting date for the scan, formatted as *"YYYY-MM-DD"*.
* **end_date**: The end date for the scan (*"YYYY-MM-DD"*). Use None to scan up to the present day.
* **outdir**: The target directory where the *CSV* files will be saved.
* **save_csv**: A boolean (*True/False*) to determine whether to export the data to a *CSV* file.
* **overwrite**: A boolean (*True/False*) to specify whether to overwrite the existing *CSV* file if it already exists.

In [ ]:
df_days, df_groups = await scan_days_with_tests_real_status(
    start_date="2023-01-01",
    end_date=None,
    outdir="~/breakaway_scan",
    save_csv=False,
    overwrite=True,
)

### Query and Data Filtering (features.py)

The `features.py` module contains all the functions required to extract *physical properties* from the M1M3 system (e.g., force, displacement, and states). Additionally, it extracts *breakaway force and displacement*, and calculates *stiffness* using a linear fit.

#### Available Arguments:

* **df_groups**: Test groups identified by scan_status.py.
* **outdir**: The target directory where the *CSV* files will be saved.
* **csv_name**: The filename for the exported *CSV*.
* **include_mount_angles**: A boolean (*True/False*) to determine whether to download elevation and azimuth angles.
* **overwrite**: A boolean (*True/False*) to specify whether to overwrite the existing *CSV* file if it already exists.

In [ ]:
df_feat = await extract_features_for_groups(
    df_groups=df_groups,
    outdir="~/breakaway_features",
    csv_name="breakaway_features_2023.csv",
    include_mount_angles=True,
    overwrite=True,
)

# display(df_feat.head())

### Plots

This optional section uses the processed **DataFrame** for visualization and further analysis.

#### Dashboard HP timeline trend (plots_bokeh.py)

This dashboard provides an *interactive* timeline trend for each hardpoint. It allows for the identification of *long-term mechanical shifts*, specifically highlighting the transition from the unstable behavior observed in 2023-2024 to the current stable regime in 2025.

In [ ]:
bokeh_1 = bokeh_daily_stiffness_dashboard(df_feat)
show(bokeh_1)

# save_dashboard_html(df_feat, outpath=os.path.expanduser("~/_static/dashboard_hardpoints.html"))

### Dashboard: Stiffness vs. Elevation Angle (plot_bokeh_gauss.py)

This dashboard, generated via `plot_bokeh_gauss.py`, illustrates how stiffness varies as a function of the elevation angle. It achieves this by applying a Normal (*Gaussian*) fit to the stiffness distribution within specific angular ranges.

#### Available Arguments:

* **df_feat**: The input DataFrame containing extracted features.
* **el_bin_w**: The width of the bins for grouping *elevation angles* (e.g., 5° bins).
* **hist_bins**: The number of bins used for the density histogram.
* **clip_percentiles**: A filter used to exclude outliers; by default, it clips data points outside the *1st and 99th percentiles*.
* **min_n_fit**: The minimum number of data points (n) required to perform a valid *Normal fit*.
* **states**: The specific *testing states* (Tension/Compression) to be included in the plot.

In [ ]:
from lsst.sitcom.tn082.plot_bokeh_gauss import bokeh_stiffness_gaussian_by_elevation

bokeh_2 = bokeh_stiffness_gaussian_by_elevation(
    df_feat,
    el_bin_w=5,
    hist_bins=40,
    clip_percentiles=(1, 99),
    min_n_fit=10,
    states=["TESTINGPOSITIVE", "TESTINGNEGATIVE"],
)

show(bokeh_2)

# save_dashboard_html(df_feat, outpath=os.path.expanduser("~/_static/dashboard_elevation.html"))

#### Breakaway Force vs. Displacement (plot_breakaway_per_hp.py)

The `plot_breakaway_per_hp.py` module provides a graphical representation of data points relative to the defined *tension and compression bands*. This visualization allows for the immediate identification of points that fall within expected limits versus those exhibiting anomalous behavior (outliers) outside these thresholds.

#### Available Arguments:

* **df_feat**: The input DataFrame containing extracted features.
* **force_col**: The column mapped to the y-axis, representing the *force measured at the moment of breakaway*.
* **disp_col**: The column mapped to the x-axis, representing the *displacement measured at the moment of breakaway*.
* **show**: A boolean (*True/False*) to determine whether to display the plots within the notebook.
* **save_png**: A boolean (*True/False*) to determine whether to export and save the plots in PNG format.

In [ ]:
from lsst.sitcom.tn082.plot_breakaway_per_hp import plot_breakaway_per_hp

plot_1 = plot_breakaway_per_hp(
    df_feat,
    force_col="breakaway_force_N",
    disp_col="breakaway_disp_um",
    show=True,
    save_png=True,
)

#### Heatmap Analysis (heatmap_per_HP.py)

The `heatmap_per_HP.py` module provides a visualization of how stiffness varies across different Hardpoints (HPs) and testing states. This format offers a clear and intuitive way to observe the temporal evolution of stiffness values, making it easier *to identify trends or anomalies over time*.

#### Available Arguments:

* **df_feat**: The input DataFrame containing extracted features.
* **states**: The specific *testing states* (Tension/Compression) to be included in the plot.
* **save_png**: A boolean (*True/False*) to determine whether to export the heatmap in PNG format.


In [ ]:
from lsst.sitcom.tn082.heatmap_per_HP import plot_monthly_heatmap_stiffness

fig_pos = plot_monthly_heatmap_stiffness(
    df_feat,
    states=["TESTINGPOSITIVE"],
    save_png=True,
)

fig_neg = plot_monthly_heatmap_stiffness(
    df_feat, states=["TESTINGNEGATIVE"], save_png=True
)

#### Historical Trend Analysis Since 2025 (plots_hp_daily.py)

The `plots_hp_daily.py` module contains two primary output functions. The first visualizes *breakaway* behavior from 2025 onwards, while the second tracks the *daily mean stiffness trends* for each hardpoint.

#### Available Arguments for Breakaway:

* **df_feat**: The input DataFrame containing extracted features.
* **since**: The starting date for the plot (e.g., *"2025-01-01"*).
* **time_col**: The DataFrame column representing the *timestamp* of the tests.
* **require_stiff_ok**: A boolean (*True/False*) to determine whether to apply the filtered stiffness criteria.
* **show_inband_markers**: A boolean (*True/False*) to toggle the visibility of data markers within the bands.
* **outdir**: The target directory where the generated *PNG* files will be saved.
* **show**: A boolean (*True/False*) to determine whether to display the plots within the notebook.

In [ ]:
from lsst.sitcom.tn082.plots_hp_daily import (
    plot_daily_stiffness_each_hp,
    plot_breakaway_each_hp,
)

plot_breakaway_each_hp(
    df_feat,
    since="2025-01-01",
    time_col="t_start_utc",
    require_stiff_ok=True,
    show_inband_markers=True,
    outdir="_static/breakaway_by_hp_2025",
    show=True,
)

#### Available Arguments for Mean Stiffness:

* **df_feat**: The input DataFrame containing extracted features.
* **since**: The starting date for the analysis (e.g., *"2025-01-01"*).
* **require_stiff_ok**: A boolean (*True/False*) to determine whether to apply the filtered stiffness criteria.
* **agg**: The statistical method for aggregation; users can select either *"mean"* or *"median"*.
* **show_std_band**: A boolean (*True/False*) to toggle the visibility of the *Standard Deviation ($\sigma$)* bands around the trend line.
* **outdir**: The target directory where the generated *PNG* files will be saved.
* **show**: A boolean (*True/False*) to determine whether to display the plots within the notebook.

In [ ]:
plot_daily_stiffness_each_hp(
    df_feat,
    since="2025-01-01",
    require_stiff_ok=True,
    agg="mean",
    show_std_band=True,
    outdir="_static/stiffness_daily_by_hp_2025",
    show=True,
)

#### Standard Deviation Analysis (plot_std_by_hp.py)

The `plot_std_by_hp.py` module visualizes the distribution of the stiffness *Standard Deviation ($\sigma$)* for each hardpoint and testing state. In addition to the graphical plots, it generates a *summary table* containing the average standard deviation per HP and state, providing a numerical baseline for system stability.

In [ ]:
from lsst.sitcom.tn082.plot_std_by_hp import plot_std_by_hp_bar

plot_std_by_hp_bar(df_feat, savepath="_scan/std_by_hp_2025.png")

#### Criteria for Valid Days

To ensure the reliability of the analysis, a valid day is defined based on the following filtering criteria:

1. **Elevation Filter**: Only data points with an elevation *angle > 20°* are included.
2. **Stiffness Filter**: Stiffness values must be > 1 N/µm to *exclude non-physical or erroneous readings*.
3. **Consistency Filter** (stiff_ok): Only records where the stiff_ok flag is True are considered.
4. **Daily Completeness Requirement**, a day is retained only if it contains:
   - Data for all 6 Hardpoints (HPs).
   - Both testing states (Tension and Compression) for every HP.

> **Note**:
> As of 2026-03-16, we have identified 154 valid days since 2025-01-01.


In [ ]:
mask = df_feat["valid_day"] & (pd.to_datetime(df_feat["date"]) >= "2025-01-01")
n_valid = df_feat.loc[mask, "date"].str[:10].nunique()
print(f"Valid days since 2025: {n_valid}")